# Lab 17-11 · Diversity, Beams and Spatial Streams

Six parts, each built on `mimo.py`. **Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import mimo as mm
rng = np.random.default_rng(1711)

## Part 1 · Combining

*Predict:* the BER at 10 dB per branch for 1, 2 and 4 branches with SC, EGC and MRC.

In [2]:
for n in (1, 2, 4):
    row = {k: mm.simulate_combining(n, 10, k, 300_000, rng) for k in ("sc", "egc", "mrc")}
    print(f"{n} branch(es): " + ", ".join(f"{k.upper()} {v:.2e}" for k, v in row.items()) + f"   (MRC formula {mm.ber_bpsk_mrc(10, n):.2e})")

1 branch(es): SC 2.28e-02, EGC 2.32e-02, MRC 2.28e-02   (MRC formula 2.33e-02)


2 branch(es): SC 2.97e-03, EGC 2.04e-03, MRC 1.65e-03   (MRC formula 1.60e-03)


4 branch(es): SC 1.30e-04, EGC 2.00e-05, MRC 6.67e-06   (MRC formula 9.70e-06)


## Part 2 · Alamouti against MRC

*Predict:* the BER of 2×1 Alamouti and 1×2 MRC at 10 dB, and the horizontal gap between their curves.

In [3]:
grid = np.arange(0, 21, 5.0)
ala = [mm.simulate_alamouti(x, 200_000, rng) for x in grid]
mrc = [mm.simulate_combining(2, x, "mrc", 400_000, rng) for x in grid]
plt.semilogy(grid, ala, "o-", label="Alamouti 2x1 (simulated)")
plt.semilogy(grid, mrc, "s-", label="MRC 1x2 (simulated)")
plt.semilogy(grid, [mm.ber_bpsk_mrc(10 ** (x / 10), 2) for x in grid], "k:", label="MRC formula")
plt.xlabel("average SNR per branch (dB)"); plt.ylabel("BER"); plt.legend(); plt.show()
print(f"at 10 dB: Alamouti {ala[2]:.2e} (MRC formula at 7 dB: {mm.ber_bpsk_mrc(10 ** 0.7, 2):.2e}), MRC {mrc[2]:.2e}")

at 10 dB: Alamouti 5.58e-03 (MRC formula at 7 dB: 5.51e-03), MRC 1.50e-03


## Part 3 · Beams and grating lobes

*Predict:* the beamwidth of 8 half-wavelength elements, and where a grating lobe appears with one-wavelength spacing steered to 30°.

In [4]:
th = np.linspace(-90, 90, 3601)
for dl, steer in ((0.5, 0), (0.5, 30), (1.0, 30)):
    plt.plot(th, 20 * np.log10(np.maximum(mm.array_factor(8, dl, th, steer), 1e-4)), label=f"d = {dl} lambda, steer {steer} deg")
plt.ylim(-40, 2); plt.xlabel("angle (deg)"); plt.ylabel("|AF| (dB)"); plt.legend(); plt.show()
print(f"HPBW broadside {mm.hpbw_deg(8, 0.5):.2f} deg, steered 30 deg {mm.hpbw_deg(8, 0.5, 30):.2f} deg")
af = mm.array_factor(8, 1.0, th, 30)
peaks = th[(af > 0.99)]
print("directions with |AF| > 0.99 for d = lambda, steer 30:", np.round(peaks[[0, -1]], 1))

HPBW broadside 12.80 deg, steered 30 deg 14.84 deg
directions with |AF| > 0.99 for d = lambda, steer 30: [-30.6  30.7]


/tmp/ipykernel_6648/943771517.py:4: UserWarning: Attempt to set non-positive ylim on a log-scaled axis will be ignored.
  plt.ylim(-40, 2); plt.xlabel("angle (deg)"); plt.ylabel("|AF| (dB)"); plt.legend(); plt.show()


## Part 4 · Capacity scaling

*Predict:* the ergodic capacity of N×N Rayleigh channels at 20 dB for N = 1, 2, 4, 8.

In [5]:
for n in (1, 2, 4, 8):
    print(f"{n}x{n}: {mm.ergodic_capacity_mc(n, n, 100, 20_000, rng):.2f} b/s/Hz")
print(f"closed forms: SISO {mm.ergodic_siso(100):.2f}, 2x2 {mm.ergodic_2x2(100):.2f}")

1x1: 5.90 b/s/Hz
2x2: 11.29 b/s/Hz
4x4: 22.17 b/s/Hz
8x8: 43.97 b/s/Hz
closed forms: SISO 5.88, 2x2 11.29


## Part 5 · Linear receivers

*Predict:* the slope of the 2×2 ZF BER curve, and what a third receive antenna does to it.

In [6]:
grid = np.arange(0, 31, 5.0)
for nr, det in ((2, "zf"), (2, "mmse"), (3, "zf")):
    ber = [mm.simulate_spatial_mux(nr, 2, x, 60_000, rng, det) for x in grid]
    plt.semilogy(grid, np.maximum(ber, 1e-6), "o-", label=f"{nr}x2 {det.upper()}")
plt.xlabel("rho (dB)"); plt.ylabel("BER (QPSK)"); plt.legend(); plt.show()

## Part 6 · Channel hardening

*Predict:* the standard deviation of $\|\mathbf h\|^2/M$ for M = 1, 4, 64, 256.

In [7]:
for m in (1, 4, 64, 256):
    g = np.sum(np.abs(mm.crandn(rng, m, 20_000)) ** 2, axis=0) / m
    print(f"M = {m:3d}: std {np.std(g):.4f} (1/sqrt(M) = {1 / math.sqrt(m):.4f})")

M =   1: std 0.9773 (1/sqrt(M) = 1.0000)
M =   4: std 0.4995 (1/sqrt(M) = 0.5000)
M =  64: std 0.1253 (1/sqrt(M) = 0.1250)


M = 256: std 0.0626 (1/sqrt(M) = 0.0625)
